In [36]:
import pandas as pd
import sqlite3
import re
import openai
import time
import nltk
from difflib import SequenceMatcher
import csv
from openai import OpenAI

openai.api_key = ""

In [38]:
context = f'''You are a helpful assistant.
You will be given a news headline and an edited version of the same headline. You will also be provided with a list of words that have been added to or removed from the original headline. 
Your goal is to label the words that have been added or removed based on their Part of Speech (POS). 
Additionally, you must analyze the changes to determine if they introduce or remove any of the following types of media bias. For each bias in the list mention if it has been added, removed, or is not relevant to the headline..

Types of Media Bias:
1. Spin (e.g., changing "protest" to "riot")
2. Unsubstantiated Claims (e.g., adding "experts say" without providing evidence)
3. Opinion Statements Presented as Fact (e.g., "The disastrous policy" instead of "The policy")
4. Sensationalism/Emotionalism (e.g., "horrifying accident" instead of "accident")
5. Mudslinging/Ad Hominem (e.g., "corrupt politician" instead of "politician")
6. Mind Reading (e.g., 'He obviously didn't care' without evidence of feelings)
7. Slant (e.g., highlighting only negative aspects of a story)
8. Flawed Logic (e.g., "If A, then B" without proper justification)
9. Bias by Omission (e.g., leaving out key details that support an alternative viewpoint)
10. Omission of Source Attribution (e.g., making claims without citing sources)
11. Bias by Story Choice and Placement (e.g., prioritizing negative news about a topic over positive news)
12. Subjective Qualifying Adjectives (e.g., "the so-called expert" instead of "the expert")
13. Word Choice (e.g., "freedom fighters" vs. "rebels")

You will be provided input in the format:
Original Headline: [Provide the original headline here]
Edited Headline: [Provide the edited headline here]
Added words: Word_Added1, Word_Added2,..
Removed words: Word_Removed1, Word_Removed2,..

Your response must be of the format:
Words Added: Word_Added1 [POS], Word_Added2 [POS],..
Words Removed: Word_Removed1 [POS], Word_Removed2 [POS],..

Bias Analysis list:
1. [type of bias] [Added/Removed/None]: The addition/removal of [specific word/phrase] introduces/removes this bias
2. [type of bias] [Added/Removed/None]: The addition/removal of [specific word/phrase] introduces/removes this bias
3. [type of bias] [Added/Removed/None]: The addition/removal of [specific word/phrase] introduces/removes this bias
.
.
.
N. [type of bias] [Added/Removed/None]: The addition/removal of [specific word/phrase] introduces/removes this bias.

Here is an example:
INPUT:
Original Headline: When it comes to inaugural crowds, does size matter?
Edited Headline: FoxNews.com - Breaking News
Added words: FoxNews.com, Breaking, News, Latest, Current, News
Removed words: When, it, comes, to, inaugural, crowds, does, size, matter, Fox

RESPONSE:
Words Added: FoxNews.com [Noun], Breaking [Noun], News [Noun], Latest [Adjective], Current [Adjective]
Words Removed: When [Adverb], it [Pronoun], comes [Verb], to [Preposition], inaugural [Adjective], crowds [Noun], does [Verb], size [Noun], matter [Noun], Fox [Noun]
Bias Analysis list:
1. Spin [None]: No spin has been introduced or removed in the edited headline.
2. Unsubstantiated Claims [Added]: The addition of "Breaking News" suggests the news is urgent and important, but does not provide evidence to support that claim.
3. Opinion Statements Presented as Fact [Added]: The edited headline contains no factual information and could be interpreted as presenting the opinion that Fox News is the best source for news. 
4. Sensationalism/Emotionalism [Added]: The addition of "Breaking News" suggests the news is urgent and important, potentially creating a sense of sensationalism or emotionalism.
5. Mudslinging/Ad Hominem [Removed]: No mudslinging or ad hominem has been introduced or removed in the edited headline.
6. Mind Reading [None]: No mind reading has been introduced or removed in the edited headline.
7. Slant [Added]: The edited headline does not contain any negative words, but its focus on "Breaking News" potentially slants the story to be more urgent and important than it might actually be.
8. Flawed Logic [None]: No flawed logic has been introduced or removed in the edited headline.
9. Bias by Omission [Added]: The edited headline contains no information about what the "Breaking News" actually is, potentially omitting key details that could support an alternative viewpoint.
10. Omission of Source Attribution [Removed]: The original headline cited the source of the article ("When it comes to"), which has been removed in the edited headline.
11. Bias by Story Choice and Placement [Added]: The choice to prioritize this headline over other potential news stories could introduce a bias by story choice and placement.
12. Subjective Qualifying Adjectives [Added]: The addition of "Breaking" and "Latest" in the edited headline are subjective and qualifying adjectives.
13. Word Choice [Removed]: The removal of "crowds" may potentially remove a bias in favor of large crowds being important or desirable.

ALWAYS RESPOND IN THIS EXACT FORMAT.'''

GPT-3.5

In [39]:
outlets = ["washpo", "nyt", "reuters", "rebel", "fox"]
prompts = {}
# put your directory_path here
directory_path = ''
for outlet in outlets:
    prompts[outlet] = pd.read_csv(f'{directory_path}/{outlet}.csv')
fox_prompts = prompts['fox']
nyt_prompts = prompts['nyt']
rebel_prompts = prompts['rebel']
reuters_prompts = prompts['reuters']
washpo_prompts = prompts['washpo']

In [ ]:
prompt_df = nyt_prompts
# put your filename here
annotation_output_file = ''
prompts = list(prompt_df['prompt'])
count = 0
prompt_responses = []
# Add your key here
client = OpenAI(api_key="")
with open(annotation_output_file, "a", newline="") as output_file:
    csv_writer = csv.writer(output_file)
    csv_writer.writerow(["prompt", "response"])  # Write header row
    for prompt in prompts:
        count = count + 1
        print(count)
        response = client.chat.completions.create(
        model="gpt-3.5-turbo-0301",
        messages=[
            {"role": "system", "content": context},
            {"role": "user", "content": prompt}
            ],
            max_tokens=1000
            )
        prompt_responses.append(response.choices[0].message.content)
        print([prompt, response.choices[0].message.content])
        csv_writer.writerow([prompt, response.choices[0].message.content])
        time.sleep(2)

1
['Original Headline: Activist Challenges a Sweeping Revision in Patent Law\nEdited Headline: Inventor Challenges a Sweeping Revision in Patent Law\nAdded words: Inventor\nRemoved words: Activist\n', 'Words Added: Inventor [Noun]\nWords Removed: Activist [Noun]\n\nBias Analysis list:\n1. Spin [None]: No spin has been introduced or removed in the edited headline.\n2. Unsubstantiated Claims [None]: No unsubstantiated claims have been introduced or removed in the edited headline.\n3. Opinion Statements Presented as Fact [None]: No opinion statements presented as fact have been introduced or removed in the edited headline.\n4. Sensationalism/Emotionalism [None]: No sensationalism/emotionalism has been introduced or removed in the edited headline.\n5. Mudslinging/Ad Hominem [None]: No mudslinging or ad hominem has been introduced or removed in the edited headline.\n6. Mind Reading [None]: No mind reading has been introduced or removed in the edited headline.\n7. Slant [None]: The edited he